In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [5]:
spark = SparkSession.builder \
        .appName("SparkTablesApp") \
        .master("local[4]") \
        .config("spark.dynamicAllocation.enabled", "false") \
        .config("spark.sql.adaptive.enabled", "false") \
        .enableHiveSupport() \
        .getOrCreate()

25/04/06 16:41:59 WARN Utils: Your hostname, luffy-Latitude-3400 resolves to a loopback address: 127.0.1.1; using 192.168.1.14 instead (on interface wlp0s20f3)
25/04/06 16:41:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/06 16:42:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/06 16:42:19 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [6]:
sc = spark.sparkContext

In [7]:
sc

<SparkContext master=local[4] appName=SparkTablesApp>

In [8]:
spark.sql("""
SHOW DATABASES
""").show()

25/04/06 16:43:17 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
25/04/06 16:43:17 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
25/04/06 16:43:21 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
25/04/06 16:43:21 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore luffy@127.0.1.1
25/04/06 16:43:21 WARN ObjectStore: Failed to get database default, returning NoSuchObjectException


+---------+
|namespace|
+---------+
|  default|
+---------+



In [10]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS TaxisDB
""")

DataFrame[]

In [11]:
spark.sql("""
SHOW DATABASES
""").show()

+---------+
|namespace|
+---------+
|  default|
|  taxisdb|
+---------+



In [12]:
yellowTaxiDF = spark.read \
                .option("header", "true") \
                .option("inferSchema", "true") \
                .csv("/home/luffy/Documents/Spark/Data/YellowTaxis_202210.csv")

In [13]:
yellowTaxiDF.write.mode("overwrite").saveAsTable("TaxisDB.YellowTaxisManaged")

25/04/06 16:46:57 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
25/04/06 16:46:57 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
25/04/06 16:46:57 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
25/04/06 16:46:57 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


In [14]:
spark.sql("""
SHOW TABLES IN TaxisDB
""").show()

+---------+------------------+-----------+
|namespace|         tableName|isTemporary|
+---------+------------------+-----------+
|  taxisdb|yellowtaxismanaged|      false|
+---------+------------------+-----------+



In [16]:
spark.sql("""
SELECT * FROM TaxisDB.YellowTaxisManaged LIMIT 10
""").show(truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2022-10-09 18:33:31 |2022-10-09 19:06:57  |1.0            |2.65         |1.0       |N                 |151         |142         |1           |21.0       |0.0  |0.5    |4.86     

In [17]:
spark.sql("""
DESCRIBE TABLE EXTENDED TaxisDB.YellowTaxisManaged
""").show()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|            VendorID|      int|   NULL|
|tpep_pickup_datetime|timestamp|   NULL|
|tpep_dropoff_date...|timestamp|   NULL|
|     passenger_count|   double|   NULL|
|       trip_distance|   double|   NULL|
|          RatecodeID|   double|   NULL|
|  store_and_fwd_flag|   string|   NULL|
|        PULocationID|      int|   NULL|
|        DOLocationID|      int|   NULL|
|        payment_type|      int|   NULL|
|         fare_amount|   double|   NULL|
|               extra|   double|   NULL|
|             mta_tax|   double|   NULL|
|          tip_amount|   double|   NULL|
|        tolls_amount|   double|   NULL|
|improvement_surch...|   double|   NULL|
|        total_amount|   double|   NULL|
|congestion_surcharge|   double|   NULL|
|         airport_fee|   double|   NULL|
|                    |         |       |
+--------------------+---------+-------+
only showing top